In [6]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
import pandas as pd


emotion_mapping = {}

emotion_mapping = {
     0: 'anger',
     1: 'fear',
     2: 'joy',
     3: 'love',
     4: 'sadness',
     5: 'surprise'
    }


print("Emotion mapping:", emotion_mapping)

Emotion mapping: {0: 'anger', 1: 'fear', 2: 'joy', 3: 'love', 4: 'sadness', 5: 'surprise'}


First, let's load the `training.csv` file and apply the emotion mapping.

In [10]:
# Load the training dataset
train_df = pd.read_csv('/content/drive/MyDrive/Oyrenme/FC_Aİ_Tasks2026/Tapşırıq 1/training.csv')

print("Original training_df head:")
display(train_df.head())

# Apply the emotion mapping to the 'label' column
train_df['emotion'] = train_df['label'].map(emotion_mapping)

print("Training_df head with emotion names:")
display(train_df.head())

print("Emotion distribution in training data:")
display(train_df['emotion'].value_counts())

Original training_df head:


,text,label
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,3
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,3


Training_df head with emotion names:


,text,label,emotion
0,i didnt feel humiliated,0,anger
1,i can go from feeling so hopeless to so damned...,0,anger
2,im grabbing a minute to post i feel greedy wrong,3,love
3,i am ever feeling nostalgic about the fireplac...,2,joy
4,i am feeling grouchy,3,love


Emotion distribution in training data:


,count
emotion,
fear,5362
anger,4666
love,2159
sadness,1937
joy,1304
surprise,572


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab') # Added this line as per the error message

# Text preprocessing function
def preprocess_text(text):
    text = text.lower() # Convert to lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Remove punctuation and numbers
    tokens = nltk.word_tokenize(text) # Tokenize text
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words] # Remove stopwords
    return ' '.join(tokens)

# Apply preprocessing to the 'text' column
train_df['processed_text'] = train_df['text'].apply(preprocess_text)

print("Training_df head with processed text:")
display(train_df.head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Training_df head with processed text:


,text,label,emotion,processed_text
0,i didnt feel humiliated,0,anger,didnt feel humiliated
1,i can go from feeling so hopeless to so damned...,0,anger,go feeling hopeless damned hopeful around some...
2,im grabbing a minute to post i feel greedy wrong,3,love,im grabbing minute post feel greedy wrong
3,i am ever feeling nostalgic about the fireplac...,2,joy,ever feeling nostalgic fireplace know still pr...
4,i am feeling grouchy,3,love,feeling grouchy


In [13]:
# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # You can adjust max_features

# Fit and transform the processed text
X_train_tfidf = tfidf_vectorizer.fit_transform(train_df['processed_text'])
y_train = train_df['emotion']

# Initialize and train the Logistic Regression model
logistic_model = LogisticRegression(max_iter=1000) # Increase max_iter for convergence
logistic_model.fit(X_train_tfidf, y_train)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


In [15]:
# Load test and validation datasets
test_df = pd.read_csv('/content/drive/MyDrive/Oyrenme/FC_Aİ_Tasks2026/Tapşırıq 1/test.csv')
val_df = pd.read_csv('/content/drive/MyDrive/Oyrenme/FC_Aİ_Tasks2026/Tapşırıq 1/validation.csv')

# Apply emotion mapping and preprocess text for test and validation data
test_df['emotion'] = test_df['label'].map(emotion_mapping)
test_df['processed_text'] = test_df['text'].apply(preprocess_text)

val_df['emotion'] = val_df['label'].map(emotion_mapping)
val_df['processed_text'] = val_df['text'].apply(preprocess_text)

# Transform text data using the *fitted* TF-IDF vectorizer
X_test_tfidf = tfidf_vectorizer.transform(test_df['processed_text'])
y_test = test_df['emotion']

X_val_tfidf = tfidf_vectorizer.transform(val_df['processed_text'])
y_val = val_df['emotion']

# Make predictions
y_pred_test = logistic_model.predict(X_test_tfidf)
y_pred_val = logistic_model.predict(X_val_tfidf)

print("\nClassification Report for Test Data:")
print(classification_report(y_test, y_pred_test))

print("\nClassification Report for Validation Data:")
print(classification_report(y_val, y_pred_val))


Classification Report for Test Data:
              precision    recall  f1-score   support

       anger       0.90      0.92      0.91       581
        fear       0.85      0.95      0.90       695
         joy       0.82      0.62      0.70       159
        love       0.88      0.84      0.86       275
     sadness       0.87      0.81      0.84       224
    surprise       0.87      0.50      0.63        66

    accuracy                           0.87      2000
   macro avg       0.86      0.77      0.81      2000
weighted avg       0.87      0.87      0.86      2000


Classification Report for Validation Data:
              precision    recall  f1-score   support

       anger       0.88      0.94      0.91       550
        fear       0.88      0.95      0.91       704
         joy       0.89      0.73      0.80       178
        love       0.90      0.85      0.87       275
     sadness       0.86      0.77      0.81       212
    surprise       0.86      0.62      0.72       